# 🎨 VecGlypher v7 — Генерация консистентных глифов

Запускайте ячейки по порядку сверху вниз. Каждую ячейку запускать кнопкой ▶ или `Shift+Enter`.

**Изменения в v7:**
- Исправлен SYSTEM_PROMPT (добавлен `<path>` для консистентности)
- Temperature=0.2 для более детерминированной генерации
- Добавлен seed для воспроизводимости

---
## Шаг 1 — Настройки
Запустите один раз в начале.

In [ ]:
import requests, json, os, base64, zipfile, glob
from IPython.display import display, HTML
from pathlib import Path

# ============================================================
# НАСТРОЙКИ — измените под себя
# ============================================================
MODEL_PATH = "/workspace/VecGlypher/saves/VecGlypher-27b-it"
SERVER_PORT = 30000
SERVER_URL  = f"http://localhost:{SERVER_PORT}/v1"
OUTPUT_DIR  = "/workspace/VecGlypher/outputs"

# Папки для референсов (проверяются по порядку)
REF_SEARCH_PATHS = [
    "/workspace/ref",
    "/workspace/VecGlypher/ref",
]
# Папки где искать zip архив с референсами
ZIP_SEARCH_PATHS = [
    "/workspace",
    "/workspace/VecGlypher",
]

# ============================================================
# ПАРАМЕТРЫ ГЕНЕРАЦИИ (как в оригинальном generate_glyph.py)
# ============================================================
# Низкая температура = более консистентные результаты
TEMPERATURE        = 0.2    # оригинал: 0.2 (было 0.7)
TOP_P              = 1.0    # оригинал: 1.0
TOP_K              = -1     # оригинал: -1 (отключён)
MAX_TOKENS         = 8192   # оригинал: 8192 (было 2048)
REPETITION_PENALTY = 1.0    # оригинал: 1.0 (было 1.05)
MIN_P              = 0.0
SEED               = 42     # Для воспроизводимости

# ============================================================
# SYSTEM_PROMPT — ИСПРАВЛЕН В v7
# ============================================================
# ВАЖНО: добавлен <path> для генерации только path-элементов!
SYSTEM_PROMPT = """You are a specialized vector glyph designer creating SVG path elements.

CRITICAL REQUIREMENTS:
- Each glyph must be a complete, self-contained <path> element, in reading order of the given text.
- Terminate each <path> element with a newline character
- Output ONLY valid SVG <path> elements
"""

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Вспомогательные функции
# ------------------------------------------------------------
def img_to_base64(path):
    """Конвертирует изображение в base64 data URL."""
    ext = Path(path).suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def find_ref_folder():
    """Ищет папку ref в стандартных местах."""
    for path in REF_SEARCH_PATHS:
        if os.path.isdir(path):
            files = [f for f in os.listdir(path) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            if files:
                return path, files
    return None, []

def find_zip():
    """Ищет zip архив с референсами в стандартных папках."""
    for folder in ZIP_SEARCH_PATHS:
        zips = glob.glob(f"{folder}/*.zip")
        if zips:
            return zips[0]
    return None

def generate_glyph_text(character, font_style):
    """Text-referenced: генерирует глиф по текстовому описанию стиля."""
    response = requests.post(
        f"{SERVER_URL}/chat/completions",
        headers={"Content-Type": "application/json", "Authorization": "dummy-key"},
        json={
            "model": MODEL_PATH,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Font design requirements: {font_style}\nText content: {character}"}
            ],
            "temperature": TEMPERATURE, 
            "top_p": TOP_P, 
            "top_k": TOP_K,
            "min_p": MIN_P, 
            "repetition_penalty": REPETITION_PENALTY,
            "seed": SEED,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": MAX_TOKENS
        },
        timeout=120
    )
    svg_paths = response.json()["choices"][0]["message"]["content"]
    full_svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="-50 -50 1100 1100" width="300" height="300">\n  <rect x="-50" y="-50" width="1100" height="1100" fill="white"/>\n  <g fill="black" stroke="none">{svg_paths}</g>\n</svg>'
    safe_style = font_style[:20].replace(' ', '_').replace(',', '')
    filename = f"{OUTPUT_DIR}/glyph_{character}_{safe_style}.svg"
    with open(filename, "w") as f:
        f.write(full_svg)
    return full_svg, filename

def generate_glyph_image(character, ref_image_paths):
    """Image-referenced: генерирует глиф по референсным изображениям."""
    user_content = []
    for img_path in ref_image_paths:
        user_content.append({
            "type": "image_url",
            "image_url": {"url": img_to_base64(img_path)}
        })
    user_content.append({
        "type": "text",
        "text": f"Text content: {character}"
    })
    response = requests.post(
        f"{SERVER_URL}/chat/completions",
        headers={"Content-Type": "application/json", "Authorization": "dummy-key"},
        json={
            "model": MODEL_PATH,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            "temperature": TEMPERATURE, 
            "top_p": TOP_P, 
            "top_k": TOP_K,
            "min_p": MIN_P, 
            "repetition_penalty": REPETITION_PENALTY,
            "seed": SEED,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": MAX_TOKENS
        },
        timeout=120
    )
    result = response.json()
    if "choices" not in result:
        print("❌ Ошибка от сервера:")
        print(json.dumps(result, indent=2))
        raise KeyError("choices")
    svg_paths = result["choices"][0]["message"]["content"]
    full_svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="-50 -50 1100 1100" width="300" height="300">\n  <rect x="-50" y="-50" width="1100" height="1100" fill="white"/>\n  <g fill="black" stroke="none">{svg_paths}</g>\n</svg>'
    filename = f"{OUTPUT_DIR}/glyph_img_{character}.svg"
    with open(filename, "w") as f:
        f.write(full_svg)
    return full_svg, filename

print("✅ Настройки загружены (v7 - консистентные параметры)")
print(f"   Модель : {MODEL_PATH}")
print(f"   Сервер : {SERVER_URL}")
print(f"   Папка  : {OUTPUT_DIR}")
print(f"   Temp   : {TEMPERATURE} (низкая для консистентности)")
print(f"   Seed   : {SEED} (для воспроизводимости)")

---
## Шаг 2 — Проверка сервера

In [ ]:
try:
    r = requests.get(f"{SERVER_URL}/models", timeout=3)
    models = r.json()
    print("✅ Сервер работает!")
    print(f"   Модель: {models['data'][0]['id']}")
except Exception as e:
    print(f"❌ Сервер недоступен: {e}")
    print("   Запустите vLLM: vllm serve /workspace/VecGlypher/saves/VecGlypher-27b-it --port 30000")

---
## Шаг 3 — Генерация одного символа (text-referenced)

In [ ]:
CHARACTER  = "A"
FONT_STYLE = "humanist sans-serif, 600 weight, calm, competent, business"

print(f"🎨 Генерируем '{CHARACTER}'...")
full_svg, filename = generate_glyph_text(CHARACTER, FONT_STYLE)
print(f"✅ Сохранено: {filename}")
display(HTML(full_svg))

---
## Шаг 4 — Генерация нескольких символов (text-referenced)
Каждый символ сохраняется сразу после генерации.

In [ ]:
CHARACTERS_BATCH = ["A", "B", "C", "D", "E"]
FONT_STYLE_BATCH = "elegant serif, thin weight, luxury, fashion"

results_html = "<div style='display:flex; flex-wrap:wrap; gap:15px; padding:10px;'>"
saved = []

for char in CHARACTERS_BATCH:
    print(f"  🎨 Генерируем '{char}'...", end=" ", flush=True)
    try:
        full_svg, filename = generate_glyph_text(char, FONT_STYLE_BATCH)
        saved.append(filename)
        svg_small = full_svg.replace('width="300" height="300"', 'width="150" height="150"')
        results_html += f"<div style='text-align:center'>{svg_small}<br><b>{char}</b></div>"
        print(f"✅ сохранён: {filename}")
    except Exception as e:
        print(f"❌ ошибка: {e}")

results_html += "</div>"
print(f"\n🏁 Готово! Сохранено {len(saved)} из {len(CHARACTERS_BATCH)} глифов")
display(HTML(results_html))

---
## Шаг 5 — Референсные изображения (image-referenced)

### Как загрузить референсы:
**Вариант А — папка:** загрузите PNG/JPG файлы в папку `/workspace/ref/` или `/workspace/VecGlypher/ref/`

**Вариант Б — ZIP архив:** положите zip файл в `/workspace/` или `/workspace/VecGlypher/` — скрипт сам распакует в папку `ref`

In [ ]:
# ------------------------------------------------------------
# Шаг 5.1 — Проверка и подготовка референсов
# ------------------------------------------------------------

ref_folder, ref_files = find_ref_folder()

if ref_folder:
    print(f"✅ Найдена папка референсов: {ref_folder}")
    print(f"   Файлов: {len(ref_files)}")
    for f in ref_files:
        print(f"   📄 {f}")
else:
    # Папки нет — ищем zip
    zip_path = find_zip()
    if zip_path:
        print(f"📦 Найден архив: {zip_path}")
        print("   Распаковываем...")
        extract_to = "/workspace/ref"
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            # Распаковываем только изображения, убираем вложенные папки
            for member in z.namelist():
                if member.lower().endswith((".png", ".jpg", ".jpeg")):
                    filename = os.path.basename(member)
                    if filename:
                        with z.open(member) as src, open(f"{extract_to}/{filename}", "wb") as dst:
                            dst.write(src.read())
        ref_folder, ref_files = find_ref_folder()
        if ref_folder:
            print(f"✅ Распаковано! Файлов: {len(ref_files)}")
            for f in ref_files:
                print(f"   📄 {f}")
        else:
            print("❌ В архиве не найдено изображений PNG/JPG")
    else:
        print("❌ Референсные изображения не найдены!")
        print("")
        print("   Загрузите файлы одним из способов:")
        print("   А) Папка: скопируйте PNG/JPG файлы в /workspace/ref/")
        print("   Б) ZIP: положите архив в /workspace/ — он распакуется автоматически")
        print("")
        print("   Команды для создания папки:")
        print("   mkdir -p /workspace/ref")
        print("   cp /путь/к/вашим/файлам/*.png /workspace/ref/")

In [ ]:
# ------------------------------------------------------------
# Шаг 5.2 — Предпросмотр референсов
# ------------------------------------------------------------

ref_folder, ref_files = find_ref_folder()

if not ref_folder:
    print("❌ Сначала загрузите референсы (Шаг 5.1)")
else:
    print(f"🖼️ Референсы из: {ref_folder}")
    html = "<div style='display:flex; flex-wrap:wrap; gap:10px; padding:10px; background:#f5f5f5;'>"
    for fname in sorted(ref_files):
        fpath = f"{ref_folder}/{fname}"
        b64 = img_to_base64(fpath)
        html += f"<div style='text-align:center'><img src='{b64}' style='height:120px; border:1px solid #ddd;'/><br><small>{fname}</small></div>"
    html += "</div>"
    display(HTML(html))

In [ ]:
# ------------------------------------------------------------
# Шаг 5.3 — Пакетная генерация по референсам (интерактивный интерфейс)
# ------------------------------------------------------------
import ipywidgets as widgets
from datetime import datetime

def make_batch_dir(batch_type):
    date_str = datetime.now().strftime("%Y-%m-%d")
    base = Path(OUTPUT_DIR)
    existing = [d for d in base.iterdir() if d.is_dir() and d.name.startswith(date_str)]
    next_num = len(existing) + 1
    folder_name = f"{date_str}_{next_num:03d}_{batch_type}"
    batch_dir = base / folder_name
    batch_dir.mkdir(parents=True, exist_ok=True)
    return str(batch_dir)

def get_output_folders():
    """Возвращает список папок с SVG файлами."""
    base = Path(OUTPUT_DIR)
    folders = []
    if base.exists():
        for d in sorted(base.iterdir()):
            if d.is_dir() and list(d.glob('*.svg')):
                folders.append(d.name)
    return folders

def generate_single(char, ref_paths, target_dir):
    """Генерирует один символ и сохраняет в target_dir."""
    user_content = []
    for img_path in ref_paths:
        user_content.append({
            "type": "image_url",
            "image_url": {"url": img_to_base64(img_path)}
        })
    user_content.append({"type": "text", "text": f"Text content: {char}"})
    response = requests.post(
        f"{SERVER_URL}/chat/completions",
        headers={"Content-Type": "application/json", "Authorization": "dummy-key"},
        json={
            "model": MODEL_PATH,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            "temperature": TEMPERATURE, 
            "top_p": TOP_P, 
            "top_k": TOP_K,
            "min_p": MIN_P, 
            "repetition_penalty": REPETITION_PENALTY,
            "seed": SEED,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": MAX_TOKENS
        },
        timeout=120
    )
    result = response.json()
    if "choices" not in result:
        raise Exception(result.get('error', {}).get('message', str(result)))
    svg_paths = result["choices"][0]["message"]["content"]
    full_svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="-50 -50 1100 1100" width="300" height="300">\n  <rect x="-50" y="-50" width="1100" height="1100" fill="white"/>\n  <g fill="black" stroke="none">{svg_paths}</g>\n</svg>'
    safe_name = char if char.isupper() else f"{char}_"
    filename = f"{target_dir}/{safe_name}.svg"
    with open(filename, "w") as f:
        f.write(full_svg)
    return full_svg, filename

# ============================================================
# ВКЛАДКИ
# ============================================================
# ------------------------------------------------------------
# Вкладка 1 — Пакетная генерация
# ------------------------------------------------------------
mode = widgets.ToggleButtons(
    options=[
        ('🔠 Весь алфавит (A-Z + a-z)', 'full'),
        ('🔡 Только заглавные (A-Z)', 'upper'),
        ('🔤 Только строчные (a-z)', 'lower'),
    ],
    description='Режим:',
    style={'description_width': 'initial'}
)

run_btn = widgets.Button(
    description='🎨 Запустить генерацию',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='40px')
)

batch_out = widgets.Output()

def on_run(b):
    with batch_out:
        batch_out.clear_output()
        ref_folder, ref_files = find_ref_folder()
        if not ref_folder:
            print("❌ Референсы не найдены! Сначала выполните Шаг 5.1")
            return
        ref_paths = [f"{ref_folder}/{f}" for f in sorted(ref_files)]

        if mode.value == 'full':
            chars = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
        elif mode.value == 'upper':
            chars = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
        else:
            chars = list('abcdefghijklmnopqrstuvwxyz')

        batch_dir = make_batch_dir(mode.value)
        print(f"📋 Режим     : {mode.label}")
        print(f"📋 Символов  : {len(chars)}")
        print(f"📋 Референсов: {len(ref_paths)}")
        print(f"📁 Папка     : {batch_dir}")
        print(f"🎲 Seed     : {SEED}")
        print(f"🌡️ Temp     : {TEMPERATURE}")
        print()

        results_html = "<div style='display:flex; flex-wrap:wrap; gap:15px; padding:10px;'>"
        saved = []

        for char in chars:
            print(f"  🎨 '{char}'...", end=" ", flush=True)
            try:
                full_svg, filename = generate_single(char, ref_paths, batch_dir)
                saved.append(filename)
                svg_small = full_svg.replace('width="300" height="300"', 'width="100" height="100"')
                results_html += f"<div style='text-align:center'>{svg_small}<br><b>{char}</b></div>"
                print(f"✅")
            except Exception as e:
                print(f"❌ {e}")

        results_html += "</div>"
        print(f"\n🏁 Сохранено {len(saved)} из {len(chars)}")
        print(f"📁 {batch_dir}")
        display(HTML(results_html))

        # Обновляем список папок в других вкладках
        folder_select.options = ['(новая папка)'] + get_output_folders()
        view_folder_select.options = get_output_folders()

run_btn.on_click(on_run)

tab1_content = widgets.VBox([mode, run_btn, batch_out])

# ------------------------------------------------------------
# Вкладка 2 — Отдельные буквы
# ------------------------------------------------------------
chars_input = widgets.Text(
    value='A,b,C',
    description='Буквы:',
    placeholder='Например: A,b,C,d или просто AbCd',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

folder_select = widgets.Dropdown(
    options=['(новая папка)'] + get_output_folders(),
    description='Папка:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

single_run_btn = widgets.Button(
    description='🎨 Генерировать',
    button_style='primary',
    layout=widgets.Layout(width='180px', height='40px')
)

single_out = widgets.Output()

def on_single_run(b):
    with single_out:
        single_out.clear_output()
        ref_folder, ref_files = find_ref_folder()
        if not ref_folder:
            print("❌ Референсы не найдены!")
            return
        ref_paths = [f"{ref_folder}/{f}" for f in sorted(ref_files)]

        # Парсим буквы — принимаем и через запятую и подряд
        raw = chars_input.value.replace(',', '').replace(' ', '')
        chars = list(raw)
        if not chars:
            print("⚠️ Введите буквы")
            return

        # Определяем папку
        if folder_select.value == '(новая папка)':
            date_str = datetime.now().strftime("%Y-%m-%d")
            base = Path(OUTPUT_DIR)
            existing = [d for d in base.iterdir() if d.is_dir() and d.name.startswith(date_str)]
            next_num = len(existing) + 1
            target_dir = str(base / f"{date_str}_{next_num:03d}_single")
            Path(target_dir).mkdir(parents=True, exist_ok=True)
        else:
            target_dir = str(Path(OUTPUT_DIR) / folder_select.value)

        print(f"📋 Буквы  : {', '.join(chars)}")
        print(f"📁 Папка  : {target_dir}")
        print()

        results_html = "<div style='display:flex; flex-wrap:wrap; gap:15px; padding:10px;'>"
        saved = []

        for char in chars:
            print(f"  🎨 '{char}'...", end=" ", flush=True)
            try:
                full_svg, filename = generate_single(char, ref_paths, target_dir)
                saved.append(filename)
                svg_small = full_svg.replace('width="300" height="300"', 'width="100" height="100"')
                results_html += f"<div style='text-align:center'>{svg_small}<br><b>{char}</b></div>"
                print(f"✅")
            except Exception as e:
                print(f"❌ {e}")

        results_html += "</div>"
        print(f"\n🏁 Сохранено {len(saved)} из {len(chars)}")
        display(HTML(results_html))

        # Обновляем списки папок
        folder_select.options = ['(новая папка)'] + get_output_folders()
        view_folder_select.options = get_output_folders()

single_run_btn.on_click(on_single_run)

tab2_content = widgets.VBox([
    chars_input,
    folder_select,
    single_run_btn,
    single_out
])

# ------------------------------------------------------------
# Вкладка 3 — Просмотр и удаление
# ------------------------------------------------------------
view_folder_select = widgets.Dropdown(
    options=get_output_folders(),
    description='Папка:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

view_btn = widgets.Button(
    description='👁️ Показать',
    button_style='info',
    layout=widgets.Layout(width='130px', height='40px')
)

refresh_view_btn = widgets.Button(
    description='🔄 Обновить',
    button_style='warning',
    layout=widgets.Layout(width='130px', height='40px')
)

delete_select = widgets.SelectMultiple(
    options=[],
    description='Удалить:',
    layout=widgets.Layout(width='400px', height='150px'),
    style={'description_width': 'initial'}
)

delete_sel_btn = widgets.Button(
    description='🗑️ Удалить выбранные',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px')
)

view_out = widgets.Output()

def refresh_view_list(folder_name):
    """Обновляет список файлов для удаления."""
    if not folder_name:
        delete_select.options = []
        return
    folder = Path(OUTPUT_DIR) / folder_name
    svgs = sorted(folder.glob('*.svg'))
    delete_select.options = [(f.stem, str(f)) for f in svgs]

def on_view(b):
    with view_out:
        view_out.clear_output()
        folder_name = view_folder_select.value
        if not folder_name:
            print("⚠️ Выберите папку")
            return
        folder = Path(OUTPUT_DIR) / folder_name
        svgs = sorted(folder.glob('*.svg'))
        if not svgs:
            print("📭 Папка пуста")
            return

        print(f"📁 {folder_name} — {len(svgs)} глифов:")
        html = "<div style='display:flex; flex-wrap:wrap; gap:10px; padding:10px; background:#f9f9f9; border-radius:8px;'>"
        for svg_file in svgs:
            content = svg_file.read_text()
            svg_small = content.replace('width="300" height="300"', 'width="90" height="90"')
            html += f"<div style='text-align:center'>{svg_small}<br><small><b>{svg_file.stem}</b></small></div>"
        html += "</div>"
        display(HTML(html))
        refresh_view_list(folder_name)

def on_refresh_view(b):
    view_folder_select.options = get_output_folders()
    folder_select.options = ['(новая папка)'] + get_output_folders()
    with view_out:
        view_out.clear_output()
        print("✅ Список обновлён")

def on_delete_selected(b):
    with view_out:
        selected = delete_select.value
        if not selected:
            print("⚠️ Выберите файлы для удаления")
            return
        for path in selected:
            try:
                Path(path).unlink()
                print(f"🗑️ Удалён: {Path(path).stem}")
            except Exception as e:
                print(f"❌ Ошибка: {e}")
        # Обновляем список
        refresh_view_list(view_folder_select.value)
        print("✅ Готово")

view_btn.on_click(on_view)
refresh_view_btn.on_click(on_refresh_view)
delete_sel_btn.on_click(on_delete_selected)

tab3_content = widgets.VBox([
    widgets.HBox([view_folder_select, widgets.VBox([view_btn, refresh_view_btn])]),
    view_out,
    widgets.HTML("<hr style='margin:10px 0'>"),
    widgets.HTML("<b>Выберите файлы для удаления:</b>"),
    delete_select,
    delete_sel_btn
])

# ============================================================
# Собираем вкладки
# ============================================================
tabs = widgets.Tab(children=[tab1_content, tab2_content, tab3_content])
tabs.set_title(0, '📦 Пакетная генерация')
tabs.set_title(1, '✏️ Отдельные буквы')
tabs.set_title(2, '👁️ Просмотр / Удаление')

display(tabs)

---
## Утилиты

### GPU статус

In [ ]:
import subprocess
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.used,memory.free,utilization.gpu --format=csv,noheader"))

---
### Сравнение параметров v6 vs v7

| Параметр | v6 (старый) | v7 (новый, как в оригинале) |
|----------|-------------|------------------------------|
| SYSTEM_PROMPT | `element` | **`<path> element`** ✅ |
| TEMPERATURE | 0.7 | **0.2** ✅ |
| TOP_P | 0.8 | **1.0** ✅ |
| TOP_K | 20 | **-1** (отключён) ✅ |
| MAX_TOKENS | 2048 | **8192** ✅ |
| REPETITION_PENALTY | 1.05 | **1.0** ✅ |
| SEED | нет | **42** ✅ |

**Почему v7 даёт более консистентные результаты:**
1. ✅ **SYSTEM_PROMPT с `<path>`** — модель генерирует только path-элементы
2. ✅ **Низкая temperature (0.2)** — меньше случайности
3. ✅ **Seed фиксирован** — одинаковые входные данные = одинаковый выход